# WWEX-1A September milestones


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

!git clone https://github.com/Break-Through-Tech/WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping.git

train = pd.read_csv('WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping/data/train.csv')
test = pd.read_csv('WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping/data/test.csv')

In [ ]:
# Load the raw data. ZIPs are read as text so we don't lose leading zeros
path = "WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping/data/"
train = pd.read_csv(path + "train.csv", dtype={"origin_zip": str, "dest_zip": str})
test = pd.read_csv(path + "test.csv", dtype={"origin_zip": str, "dest_zip": str})

# The correct spellings we want to end up with
carriers = ["Falcon Post", "Northstar Parcel", "Summit Express",
            "Ironclad Freight", "Vela Freight", "Cobalt Regional"]
services = ["Ground", "Expedited", "TwoDay", "Overnight", "LTL Standard", "LTL Guaranteed"]

# Lookup: lowercase version -> correct spelling (e.g. "falcon post" -> "Falcon Post")
carrier_map = {n.lower(): n for n in carriers}
service_map = {n.lower(): n for n in services}

# Columns that have missing or messy numbers
fill_cols = ["weight_lbs", "length_in", "width_in", "height_in"]

# Steps 1-3 run on both train and test
for df in (train, test):
    # 1. Fix messy text: remove extra spaces, make lowercase, then swap in the correct spelling
    df["carrier"] = df["carrier"].str.strip().str.lower().map(carrier_map)
    df["service_level"] = df["service_level"].str.strip().str.lower().map(service_map)

    # 2. Fix ZIPs: put back the leading zero that got lost (2108 -> 02108)
    for c in ["origin_zip", "dest_zip"]:
        df[c] = df[c].str.strip().str.zfill(5)

    # 3. Turn date text into real dates so we can sort by time later
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["ship_date"] = pd.to_datetime(df["ship_date"])

# 4. Remove duplicate rows (done after the text cleanup so near-duplicates get caught too)
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)

for df in (train, test):
    # 5a. A weight of 0 isn't real, so treat it as missing
    df.loc[df["weight_lbs"] == 0, "weight_lbs"] = np.nan

    # 5b. Find weights that are way too big (these were grams typed in as pounds)
    grams = ((df["shipment_mode"] == "parcel") & (df["weight_lbs"] > 150)) | \
            ((df["shipment_mode"] == "ltl_freight") & (df["weight_lbs"] > 20000))

    # 5c. Convert those from grams to pounds
    df.loc[grams, "weight_lbs"] = df.loc[grams, "weight_lbs"] / 453.592

    # 6. Mark which values were missing BEFORE we fill them in
    for c in fill_cols:
        df[c + "_na"] = df[c].isnull()

# 7. Fill missing values with the typical (median) value for that shipment type
# Medians come from train only, so test doesn't leak into our results
meds = train.groupby("shipment_mode")[fill_cols].median()
print(meds)
for df in (train, test):
    for c in fill_cols:
        df[c] = df[c].fillna(df["shipment_mode"].map(meds[c]))

# Checks: make sure everything worked
print(len(train), len(test))                        # expect about 39,949 and 10,114
print(train[fill_cols].isnull().sum().sum(), test[fill_cols].isnull().sum().sum())   # expect 0 0
print(train.groupby("shipment_mode")[fill_cols].describe().T)   # weights should look realistic now

# Date ranges (we need these to decide how to split the data)
print("train:", train["ship_date"].min().date(), "to", train["ship_date"].max().date())
print("test: ", test["ship_date"].min().date(), "to", test["ship_date"].max().date())

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
# Histogram of Transit Days distribution
sns.histplot(data=train, x = "transit_days" )
plt.title("Distribution of Transit Days")
plt.xlabel("Transit Days")
plt.ylabel("Frequency")

In [ ]:
# Plot of counts for on-time vs. late shipments
sns.countplot(data=train, x = "delivered_on_time" )
plt.title("Count of On-time vs. Late Shipments")
plt.xlabel("On-time vs. Late")
plt.ylabel("Count")

plt.xticks(ticks=[0, 1], labels=["Late", "On-time"])

In [ ]:
# Scatterplot for Transit Days vs. Distance Miles by Shipment Mode
sns.scatterplot(data=train.sample(100), x = "distance_miles", y = "transit_days", hue = "shipment_mode")

plt.title("Transit Days vs Distance Miles")
plt.xlabel("Distance (miles)")
plt.ylabel("Transit Days")

In [ ]:
# Boxplot of Transit Days vs. Service Level
level = ['Ground', 'Expedited', 'TwoDay', 'Overnight', 'LTL Standard', 'LTL Guaranteed']
sns.boxplot(data = train, x = 'service_level', y = 'transit_days', order = level)
plt.title("Transit Days vs Service Level")
plt.xlabel("Service Level")
plt.ylabel("Transit Days")
plt.xticks(rotation=45)

In [ ]:
# Scatterplot of Carrier Cost vs. Speed of Shipment
carrier_data = train.groupby('carrier').agg(
    mean_transit=('transit_days','mean'),
    on_time_rate=('delivered_on_time','mean'),
    mean_cost=('carrier_cost_usd','mean')
)

sns.scatterplot(data=carrier_data, x='mean_cost', y='mean_transit', hue='carrier')
plt.title("Carrier Cost vs Speed of Shipment")
plt.show()

In [ ]:
# Barplot of Average Transit Days by Month
sns.barplot(data=train, x='ship_month', y='transit_days', estimator='mean')
plt.title('Average Transit Days by Month')
plt.xlabel("Month")
plt.ylabel("Average Transit Days")

months = ["January", "February", "March", "April", "May", "June", "July", "August",
"September", "October", "November", "December"]
plt.xticks(
    ticks=range(12),
    labels=months
)
plt.xticks(rotation=30)

In [ ]:
# Barplot of Average Transit Days by Week
sns.barplot(data=train, x='ship_dow', y='transit_days', estimator='mean')
plt.title('Average Transit Days by Day of Week')
plt.xlabel("Day of Week")
plt.ylabel("Average Transit Days")

week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plt.xticks(
    ticks=range(7),
    labels=week
)
plt.xticks(rotation=30)

In [ ]:
from sklearn.metrics import mean_absolute_error  # average size of our mistakes, in days

# sort shipments by date
train = train.sort_values("ship_date").reset_index(drop=True)

# split on a DATE, not a row number, so no calendar day lands in both train and validation
split_date = train["ship_date"].quantile(0.8, interpolation="nearest").normalize()

# older shipments for training, newest shipments for validation
tr = train[train["ship_date"] < split_date].copy()
val = train[train["ship_date"] >= split_date].copy()
assert tr["ship_date"].max() < val["ship_date"].min()   # fails loudly if a day ever appears on both sides

# check the date ranges and row counts
print("tr: ", tr["ship_date"].min().date(), "to", tr["ship_date"].max().date(), len(tr))
print("val:", val["ship_date"].min().date(), "to", val["ship_date"].max().date(), len(val))

# use promised days as the prediction and measure the average error
print("Naive MAE:", mean_absolute_error(val["transit_days"], val["promised_days"]))

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor  # the model for predicting a number
from sklearn.metrics import mean_absolute_error, mean_squared_error  # ways to measure errors

#choose which columns the model can use
#remove the answer, columns that reveal the answer, and columns that are just labels
drop = ["transit_days", "delivered_on_time", "shipment_id",
        "order_date", "ship_date", "origin_zip", "dest_zip"]

#use everything except the dropped columns as features
features = [c for c in train.columns if c not in drop]

#create feature tables for training and validation
X_tr, X_val = tr[features].copy(), val[features].copy()

#tell the model which text columns are categories
for c in X_tr.select_dtypes(include="object").columns:
    cats = pd.CategoricalDtype(X_tr[c].unique())
    X_tr[c] = X_tr[c].astype(cats)
    X_val[c] = X_val[c].astype(cats)

#convert True/False columns to 1/0
for c in X_tr.select_dtypes(include="bool").columns:
    X_tr[c] = X_tr[c].astype(int)
    X_val[c] = X_val[c].astype(int)

#create the model
model = HistGradientBoostingRegressor(
    loss="absolute_error", categorical_features="from_dtype", random_state=42
)

# train the model using the older shipments
model.fit(X_tr, tr["transit_days"])

#predict transit days for the newer shipments
pred = model.predict(X_val)

#compare the predictions to the actual transit days
naive = mean_absolute_error(val["transit_days"], val["promised_days"])  #carrier's promise
mae = mean_absolute_error(val["transit_days"], pred)                    #model's average error
rmse = mean_squared_error(val["transit_days"], pred) ** 0.5             #gives more weight to big errors
within1 = (abs(val["transit_days"] - pred) <= 1).mean()                 #predictions within 1 day

print("Naive MAE (promised days):", round(naive, 3))
print("Model MAE:", round(mae, 3))
print("Model RMSE:", round(rmse, 3))
print("Within +/-1 day:", round(within1 * 100, 1), "%")